# Wyoming — Title 26 (Insurance Code) → `data/wyoming/ins_codes/*.md`

Wyoming’s **insurance** statutes are **Title 26 — Insurance Code** of the **Wyoming Statutes**. On **Justia**, the crawl root is **[`/codes/wyoming/title-26/`](https://law.justia.com/codes/wyoming/title-26/)**; sections live under **`…/title-26/chapter-…/article-…/section-26-…`** (e.g. **`…/chapter-1/article-1/section-26-1-101/`**).

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`**.

**Discovery:** BFS from the Title 26 index, following only paths under **`/codes/wyoming/title-26/`** that are **not** section pages, skipping **`appendix`**, **`chronological-history`**, and **`title-notes`**; collect every section link (~**1,072** sections across ~**127** index pages).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`WY_sec_<slug>.md`** where **`<slug>`** is the segment after **`section-`** (e.g. `26-1-101` → `WY_sec_26_1_101.md`). Display uses **`Wyo. Stat. §`** plus the slug (e.g. **`Wyo. Stat. § 26-1-101`**, aligned with Justia’s **WY Stat §** form).

Config: **MAX_SECTIONS** (**0** = all), **MAX_DISCOVERY_PAGES** (**0** = no cap). **REUSE_DISCOVERED_URLS** skips discovery when **`_wy_title26_section_urls.txt`** exists.

Run with the **`ins_ipynb/`** directory as cwd. Then **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/wyoming/title-26"
TITLE_INDEX = f"{BASE}{PATH_PREFIX}/"

OUT_DIR = Path("data") / "wyoming" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_wy_title26_section_urls.txt"
REUSE_DISCOVERED_URLS = True

SKIP_PATH_SUBSTR = ("appendix", "chronological-history", "title-notes")


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def skip_path(p: str) -> bool:
    low = p.lower()
    return any(s in low for s in SKIP_PATH_SUBSTR)


def discover_section_urls() -> list[str]:
    """BFS title-26 index + chapter/article pages; collect section URLs."""
    from collections import deque

    pref = PATH_PREFIX.lower()
    start = TITLE_INDEX
    seen: set[str] = set()
    in_q: set[str] = {path_key(start).lower()}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    fetches = 0
    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url).lower()
        in_q.discard(pk)
        if pk in seen:
            continue
        if "/section-" in pk:
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        if fetches % 25 == 0:
            print(f"… discovery fetch {fetches}, queue={len(q)}, sections={len(sections)}")
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"])
            p = path_key(absu).lower()
            if not p.startswith(pref):
                continue
            if skip_path(p):
                continue
            if "/section-" in p:
                sections.add(BASE + p + "/")
            else:
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")
    return sorted(sections, key=lambda u: label_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    path = path_key(url)
    low = path.lower()
    key = "/section-"
    i = low.rfind(key)
    if i < 0:
        raise ValueError(f"not a section URL: {url!r}")
    return path[i + len(key) :]


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_display_citation(label: str) -> str:
    """Justia slug 26-1-101 → Wyo. Stat. § 26-1-101"""
    return f"Wyo. Stat. § {label}"


def label_to_filename(label: str) -> str:
    safe = re.sub(r"[^0-9a-zA-Z]+", "_", label).strip("_").lower()
    return f"WY_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    disclaimer_skip = False
    for line in lines:
        s = line.strip()
        if not s:
            if disclaimer_skip:
                disclaimer_skip = False
            if not skip_until_substantive:
                out.append("")
            continue
        if disclaimer_skip:
            continue
        if s.startswith("Disclaimer:"):
            disclaimer_skip = True
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s in {"of", "this Section"}:
            continue
        if "American Association of Law Libraries Universal Citation Guide" in s:
            continue
        if s.startswith("necessarily the official citation"):
            continue
        if s.startswith("20") and ("Wyoming Stat" in s or "WY Stat" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("Wyoming Statutes") and "Title 26" in s:
            continue
        if re.match(r"^WY Stat § .+\(\s*20\d{2}\s*\)\s*$", s):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_title_26() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs under Title 26")
        all_urls = sorted(found, key=lambda u: label_sort_key(section_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        disp = label_to_display_citation(label)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"Wyoming Statutes {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**Wyoming Statutes — Title 26 (Insurance Code)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [Wyoming Legislature — Statutes TOC](https://www.wyoleg.gov/Nomenclature/StatutesTOC.aspx)\n\n"
                    f"**Section (URL slug):** {label}\n\n"
                    f"**Citation (display):** {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title_26()


… discovery fetch 25, queue=85, sections=105
… discovery fetch 50, queue=75, sections=404
… discovery fetch 75, queue=52, sections=650
… discovery fetch 100, queue=27, sections=867
… discovery fetch 125, queue=2, sections=1050
Discovered 1072 section URLs under Title 26
… 200/1072 (wrote=200 skipped=0 failed=0)
… 400/1072 (wrote=400 skipped=0 failed=0)
… 600/1072 (wrote=600 skipped=0 failed=0)
… 800/1072 (wrote=800 skipped=0 failed=0)
… 1000/1072 (wrote=1000 skipped=0 failed=0)
Done. wrote=1072 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/wyoming/ins_codes


{'wrote': 1072, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
